In [2]:
# fix working directory only once
if not "working_directory_corrected" in vars():
    %cd ..
    working_directory_corrected = True

/Users/tomseidel/Desktop/CODE/JEPAHeuristics


In [11]:
import os

# macOS-only fix for a torch/xgboost conflict (not needed on the Spark)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import copy
import random
import types

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gymnasium as gym
from huggingface_hub import snapshot_download
from stable_worldmodel.envs.two_room import TwoRoomEnv
from stable_worldmodel.solver.cem import CEMSolver
from stable_worldmodel.policy import WorldModelPolicy, PlanConfig

import utils.load_fix_pretrained as load_util
from utils.embedding_probing.experiments.shared.normalization import normalise_images
from utils.embedding_probing.experiments.shared.env_helpers import _render_agent_at
from utils.embedding_probing.models.xgboost_model import load_xgboost
from utils.embedding_probing.models.mlp.persistence import load_torch_mlp
from utils.embedding_probing.models.mlp.train import predict_torch_mlp

## CEM Cost-Function Benchmark

This notebook compares four cost functions for the CEM planner, with everything else held fixed
(same CEM solver, same checkpoint, same action-encoding/history/action-clipping fixes):

1. **Baseline** - the unmodified LeWM default cost: Squared Distance between both Embeddings
2. **True-distance heuristic** - our trained XGBoost/MLP models which predict the true distance for a   fixed wall configuration
3. **Position-probing + Euclidean** - decode (x, y) from each embedding with our position-probing model(s), then take the euclidean distance between the two decoded points (ignoring the wall)
4. **Position-probing + maze formula** - decode (x, y) from each embedding with our position-probing model(s) but now using our wall aware formula

All four conditions are run on the **same fixed set of (start, goal) episodes** (paired comparison) and
compared on success rate and steps-to-goal.

In [4]:
DEVICE = "cpu"

N_EPISODES = 100  # amount of runs per CEM function
MAX_STEPS = 50 # prevent infinite steps
SUCCESS_DIST = 16.0 # if dist < SUCCESS_DIST => goal reached
EPISODE_SEED_BASE = 123  # fixed base seed -> identical (start, goal) pairs across all four conditions

### Load the checkpoint once

We load the base checkpoint a single time and `copy.deepcopy` it per condition below, so each cost
function gets its own independent model.

In [5]:
checkpoint_path = snapshot_download(
    repo_id="quentinll/lewm-tworooms",
    revision="77adaae0bc31deab21c93740d1f8bb947cd0bdec",
)
base_model = load_util.load_fix_pretrained(checkpoint_path)
base_model.eval()

print("predictor history size:", base_model.predictor.num_frames)
print("action_encoder expects:", base_model.action_encoder.input_dim, "dims")

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 770.59it/s]

15:06:54 | INFO  | utils.py    | Loading checkpoint from folder /Users/tomseidel/.cache/huggingface/hub/models--quentinll--lewm-tworooms/snapshots/77adaae0bc31deab21c93740d1f8bb947cd0bdec...
15:06:54 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
predictor history size: 3
action_encoder expects: 10 dims


### Shared fixes, applied to every condition

These three patches are engineering requirements for this checkpoint/env combination, not part of what
we are comparing - they are applied identically to all four conditions:

- **Action-encoder tiling**: checkpoint expects 10D actions, `TwoRoomEnv` only gives 2D so we need to re-shape
- **Action clipping in `get_cost`**: CEM samples actions with no awareness of the env's
  `[-1, 1]` action bounds so we need to clip the action bounds to the possible movements.
- **History buffer**: predictor needs 3 history frames as input

In [6]:
class TiledActionEncoder(nn.Module):
    """Tiles a 2D action to the action_width of the embedder: 10D"""

    def __init__(self, inner, action_width=10):
        super().__init__()
        self.inner = inner
        self.action_width = action_width

    def forward(self, act):
        reps = self.action_width // act.shape[-1]
        wide = act.repeat(*([1] * (act.ndim - 1)), reps)
        return self.inner(wide)


def build_model(criterion_fn=None):
    """
    creates an independent copy of the base checkpoint with added action clipping
    """
    model = copy.deepcopy(base_model)
    model.eval()

    model.action_encoder = TiledActionEncoder(model.action_encoder)

    if criterion_fn is not None:
        model.criterion = types.MethodType(criterion_fn, model)

    original_get_cost = model.get_cost

    def clipped_get_cost(self, info_dict, action_candidates):
        action_candidates = torch.clamp(action_candidates, -1.0, 1.0)
        return original_get_cost(info_dict, action_candidates)

    model.get_cost = types.MethodType(clipped_get_cost, model)
    return model

In [ ]:
def run_episode(policy, vec_env, seed, max_steps=MAX_STEPS, success_dist=SUCCESS_DIST):
    """
    A single episode: one model's single attempt to guide the agent to the goal
    """
    raw_env = vec_env.envs[0] # extract TwoRoomEnv
    obs, info = vec_env.reset(seed=seed) # Reset TwoRoomEnv with fixed seed: every CEM should have the same agent and target position

    agent_pos = info["state"][0]
    goal_pos = info["goal_state"][0]
    start_dist = float(info["distance_to_target"][0])

    first_frame = _render_agent_at(raw_env, agent_pos) # create image pixels 
    history = [first_frame, first_frame, first_frame]  # bootstrap 3-frame history: predictor needs 3 frames min

    saved_pos = raw_env.agent_position.clone() # aget position backup
    goal_frame = _render_agent_at(raw_env, goal_pos) # create image pixels
    raw_env.agent_position = saved_pos

    for step in range(max_steps):
        pixels = torch.from_numpy(np.stack(history)).float().unsqueeze(0)             # [[[H,W,C], [H,W,C], [H,W,C]]]
        goal_pixels = torch.from_numpy(goal_frame).float().unsqueeze(0).unsqueeze(0)  # [[[H,W,C]]]

        info_dict = {
            "pixels": pixels,
            "goal": goal_pixels,
            "action": torch.zeros(1, 3, 2), # boilerplate for lib
        }

        action = policy.get_action(info_dict) # best next action
        obs, reward, terminated, truncated, info = vec_env.step(action) # run action

        agent_pos = info["state"][0] # updated agent position after action
        history = history[1:] + [_render_agent_at(raw_env, agent_pos)] # pop last and add new 

        dist = float(info["distance_to_target"][0]) # new euclidean distance after action

        if dist < success_dist: # return when distance to goal small enough
            return {"seed": seed, "start_dist": start_dist, "success": True, "steps": step + 1, "final_dist": dist}

    return {"seed": seed, "start_dist": start_dist, "success": False, "steps": max_steps, "final_dist": dist}


def run_benchmark(condition_name, model, seeds, num_samples=100, n_steps=10, topk=10):
    """
    Builds a fresh solver/policy/env around model and runs run_episode over every seed.
    """
    cfg = PlanConfig(
        horizon=8, # history + receding
        receding_horizon=5, # 5 future steps
        history_len=3, # 3 history frames as input
        action_block=1, 
        )
    solver = CEMSolver(
        model=model, # worldmodel with flexible costfunction
        batch_size=1, # single environment
        num_samples=num_samples, # 100
        n_steps=n_steps, #10
        topk=topk, # top x from num_samples
        device=DEVICE, # cuda / cpu
        seed=0, # fixed seed 
    )
    policy = WorldModelPolicy(
        solver=solver, # CEM solver from above
        config=cfg, # Plan config from above
        transform={"pixels": normalise_images, "goal": normalise_images}, # normalize inputs 
    )
    vec_env = gym.vector.SyncVectorEnv([lambda: TwoRoomEnv()]) # set TwoRoomEnv
    policy.set_env(vec_env)

    rows = []
    for i, seed in enumerate(seeds): # loop from 0-99
        result = run_episode(policy, vec_env, seed) # setup TwoRoomEnv with seed = seed and policy from above 
        result["condition"] = condition_name # baseline | true distance ...
        rows.append(result)
        if (i + 1) % 10 == 0: # update every 10 episodes
            print(f"[{condition_name}] {i + 1}/{len(seeds)} episodes done")

    return pd.DataFrame(rows)


episode_seeds = [EPISODE_SEED_BASE + i for i in range(N_EPISODES)]

## Condition 1: Baseline (unmodified LeWM default cost)

No `criterion_fn` override - keeps LeWM's own squared-embedding-distance cost. Still gets the shared
action-tiling/clipping fixes from `build_model`, since those are prerequisites for this checkpoint to
run at all, not part of what we are testing here.

In [ ]:
baseline_model = build_model(criterion_fn=None)
baseline_results = run_benchmark("baseline", baseline_model, episode_seeds)
baseline_results.to_pickle("data/cem_benchmark_baseline.pkl")
baseline_results["success"].mean() # prob to succeed

CEM solve time: 1.4469 seconds
CEM solve time: 1.3481 seconds
CEM solve time: 1.4878 seconds
CEM solve time: 1.4647 seconds
CEM solve time: 1.5769 seconds
CEM solve time: 1.5779 seconds
CEM solve time: 1.5044 seconds
CEM solve time: 1.5737 seconds
CEM solve time: 1.5862 seconds
CEM solve time: 1.5823 seconds
CEM solve time: 1.6088 seconds
CEM solve time: 1.4161 seconds
CEM solve time: 1.4078 seconds
CEM solve time: 1.6307 seconds
CEM solve time: 1.6546 seconds
CEM solve time: 1.4839 seconds
CEM solve time: 1.4490 seconds
CEM solve time: 1.5379 seconds
CEM solve time: 1.7162 seconds
CEM solve time: 1.6413 seconds
CEM solve time: 1.7209 seconds
CEM solve time: 1.4551 seconds
CEM solve time: 1.4542 seconds
CEM solve time: 1.5513 seconds
CEM solve time: 1.5289 seconds
CEM solve time: 1.6493 seconds
CEM solve time: 1.5993 seconds
CEM solve time: 1.6548 seconds
CEM solve time: 1.5582 seconds
CEM solve time: 1.5502 seconds
CEM solve time: 1.5282 seconds
CEM solve time: 1.5173 seconds
CEM solv

np.float64(0.23)

## Shared

In [ ]:
def xgb_predict_fn(model):
    """
    Wraps a loaded XGBoost(/MultiOutputRegressor) model as a plain X -> y_pred function.
    """
    return lambda X: model.predict(X)


def mlp_predict_fn(model, device="cpu"):
    """
    Wraps a loaded TorchMLP model as a plain X -> y_pred function.
    """
    return lambda X: predict_torch_mlp(model, X, device=device)

## Condition 2: True-distance heuristic

We compare both model types we trained on the
same true_distance dataset (XGBoost and MLP) against each other, not just against the baseline.

In [ ]:
def make_true_distance_criterion(predict_fn):
    """
    Predicts wall-aware distance directly from the embedding pair.
    Compares both MLP and XGB
    """

    def true_distance_criterion(self, info_dict):
        pred_emb = info_dict["predicted_emb"]  # Plan iteration of 5 steps
        goal_emb = info_dict["goal_emb"]       # (B, S, T, dim)

        pred_final = pred_emb[..., -1, :] # extract latest of the 5 steps (current position)
        goal_final = goal_emb[..., -1, :].expand_as(pred_final) # goal embedding broadcasted 

        B, S, D = pred_final.shape
        X = torch.cat([pred_final, goal_final], dim=-1).reshape(B * S, 2 * D) # Input for our model
        costs_np = predict_fn(X.detach().cpu().numpy())
        return torch.from_numpy(costs_np).to(pred_emb.device, dtype=pred_emb.dtype).reshape(B, S)

    return true_distance_criterion


TRUE_DISTANCE_XGB_PATH = "models/true_distance/2026-11-02_20-12-55_xgboost.json"
TRUE_DISTANCE_MLP_PATH = "models/true_distance/2026-11-02_20-12-55_mlp"

true_distance_xgb = load_xgboost(TRUE_DISTANCE_XGB_PATH, device="cpu")
true_distance_mlp = load_torch_mlp(TRUE_DISTANCE_MLP_PATH, device="cpu")

In [16]:
true_distance_xgb_model = build_model(criterion_fn=make_true_distance_criterion(xgb_predict_fn(true_distance_xgb)))
true_distance_xgb_results = run_benchmark("true_distance_xgb", true_distance_xgb_model, episode_seeds)
true_distance_xgb_results.to_pickle("data/cem_benchmark_true_distance_xgb.pkl")
true_distance_xgb_results["success"].mean()

CEM solve time: 1.4166 seconds
CEM solve time: 1.2994 seconds
CEM solve time: 1.3837 seconds
CEM solve time: 1.4906 seconds
CEM solve time: 1.4615 seconds
CEM solve time: 1.4810 seconds
CEM solve time: 1.4136 seconds
CEM solve time: 1.5514 seconds
CEM solve time: 1.5205 seconds
CEM solve time: 1.4658 seconds
CEM solve time: 1.4662 seconds
CEM solve time: 1.4570 seconds
CEM solve time: 1.4952 seconds
CEM solve time: 1.3455 seconds
CEM solve time: 1.3239 seconds
CEM solve time: 1.2838 seconds
CEM solve time: 1.4504 seconds
CEM solve time: 1.4036 seconds
CEM solve time: 1.4768 seconds
CEM solve time: 1.4822 seconds
CEM solve time: 1.4805 seconds
CEM solve time: 1.5098 seconds
CEM solve time: 1.5285 seconds
CEM solve time: 1.5876 seconds
CEM solve time: 1.5458 seconds
CEM solve time: 1.4562 seconds
CEM solve time: 1.4773 seconds
CEM solve time: 1.4806 seconds
CEM solve time: 1.4864 seconds
CEM solve time: 1.5026 seconds
CEM solve time: 1.5406 seconds
CEM solve time: 1.4273 seconds
CEM solv

np.float64(0.37)

In [17]:
true_distance_mlp_model = build_model(criterion_fn=make_true_distance_criterion(mlp_predict_fn(true_distance_mlp)))
true_distance_mlp_results = run_benchmark("true_distance_mlp", true_distance_mlp_model, episode_seeds)
true_distance_mlp_results.to_pickle("data/cem_benchmark_true_distance_mlp.pkl")
true_distance_mlp_results["success"].mean()

CEM solve time: 1.4375 seconds
CEM solve time: 1.2913 seconds
CEM solve time: 1.3080 seconds
CEM solve time: 1.4020 seconds
CEM solve time: 1.5195 seconds
CEM solve time: 1.3826 seconds
CEM solve time: 1.3350 seconds
CEM solve time: 1.3121 seconds
CEM solve time: 1.3008 seconds
CEM solve time: 1.2804 seconds
CEM solve time: 1.3540 seconds
CEM solve time: 1.4492 seconds
CEM solve time: 1.4362 seconds
CEM solve time: 1.4760 seconds
CEM solve time: 1.4454 seconds
CEM solve time: 1.4917 seconds
CEM solve time: 1.4943 seconds
CEM solve time: 1.2985 seconds
CEM solve time: 1.3769 seconds
CEM solve time: 1.3962 seconds
CEM solve time: 1.4236 seconds
CEM solve time: 1.4410 seconds
CEM solve time: 1.4579 seconds
CEM solve time: 1.5152 seconds
CEM solve time: 1.4700 seconds
CEM solve time: 1.4594 seconds
CEM solve time: 1.4605 seconds
CEM solve time: 1.4799 seconds
CEM solve time: 1.4764 seconds
CEM solve time: 1.4768 seconds
CEM solve time: 1.5023 seconds
CEM solve time: 1.5360 seconds
CEM solv

np.float64(0.35)

## Condition 3: Position-probing + Euclidean distance

Decode (x, y) separately from `pred_final` and `goal_final` with our trained position-probing
regressor, then score candidates by the Euclidean distance between the two decoded points. This
ignores the wall entirely 

In [ ]:
def make_position_euclidean_criterion(predict_fn):
    """
    Decodes (x, y) from the predicted and goal embeddings 
    and calculates euclidean distance 
    """

    def position_euclidean_criterion(self, info_dict):
        pred_emb = info_dict["predicted_emb"][..., -1, :]
        goal_emb = info_dict["goal_emb"][..., -1, :].expand_as(pred_emb)

        B, S, D = pred_emb.shape
        pred_pos = predict_fn(pred_emb.detach().cpu().numpy().reshape(B * S, D))
        goal_pos = predict_fn(goal_emb.detach().cpu().numpy().reshape(B * S, D))

        dist = np.linalg.norm(pred_pos - goal_pos, axis=-1)
        return torch.from_numpy(dist).to(pred_emb.device, dtype=pred_emb.dtype).reshape(B, S)

    return position_euclidean_criterion


POSITION_XGB_PATH = "models/position_probing/2026-11-17_20-53-47_xgboost.json"
POSITION_MLP_PATH = "models/position_probing/2026-11-17_20-53-47_mlp"

position_xgb = load_xgboost(POSITION_XGB_PATH, device="cpu")
position_mlp = load_torch_mlp(POSITION_MLP_PATH, device="cpu")


In [ ]:
position_euclidean_xgb_model = build_model(criterion_fn=make_position_euclidean_criterion(xgb_predict_fn(position_xgb)))
position_euclidean_xgb_results = run_benchmark("position_euclidean_xgb", position_euclidean_xgb_model, episode_seeds)
position_euclidean_xgb_results.to_pickle("data/cem_benchmark_position_euclidean_xgb.pkl")
position_euclidean_xgb_results["success"].mean()

In [ ]:
position_euclidean_mlp_model = build_model(criterion_fn=make_position_euclidean_criterion(mlp_predict_fn(position_mlp)))
position_euclidean_mlp_results = run_benchmark("position_euclidean_mlp", position_euclidean_mlp_model, episode_seeds)
position_euclidean_mlp_results.to_pickle("data/cem_benchmark_position_euclidean_mlp.pkl")
position_euclidean_mlp_results["success"].mean()

## Condition 4: Position-probing + wall-aware maze formula

Same decoded positions as Condition 3, but combined with the exact formula used to generate the
true_distance training labels (`true_distance_batch`) instead of plain Euclidean distance. The door
is respected in this implementation

In [ ]:
from utils.embedding_probing.experiments.true_distance.generate_true_distance_dataset import (
    true_distance_batch,
)


def _wall_geometry(env):
    return {
        "axis": env.wall_axis,
        "center": float(env.WALL_CENTER),
        "door_positions": env.door_positions[: env.num_doors].tolist(),
        "door_sizes": env.door_sizes[: env.num_doors].tolist(),
    }


# door layout is fixed
_geom_env = TwoRoomEnv()
_geom_env.reset()
MAZE_GEOMETRY = _wall_geometry(_geom_env)


def make_position_formula_criterion(predict_fn, geom):
    """
    Decode position of agent and goal and calculate wall aware distance
    """

    def position_formula_criterion(self, info_dict):
        pred_emb = info_dict["predicted_emb"][..., -1, :]
        goal_emb = info_dict["goal_emb"][..., -1, :].expand_as(pred_emb)

        B, S, D = pred_emb.shape
        pred_pos = predict_fn(pred_emb.detach().cpu().numpy().reshape(B * S, D))
        goal_pos = predict_fn(goal_emb.detach().cpu().numpy().reshape(B * S, D))

        dist = true_distance_batch(pred_pos, goal_pos, geom)
        return torch.from_numpy(dist).to(pred_emb.device, dtype=pred_emb.dtype).reshape(B, S)

    return position_formula_criterion


In [ ]:
position_formula_xgb_model = build_model(criterion_fn=make_position_formula_criterion(xgb_predict_fn(position_xgb), MAZE_GEOMETRY))
position_formula_xgb_results = run_benchmark("position_formula_xgb", position_formula_xgb_model, episode_seeds)
position_formula_xgb_results.to_pickle("data/cem_benchmark_position_formula_xgb.pkl")
position_formula_xgb_results["success"].mean()

In [ ]:
position_formula_mlp_model = build_model(criterion_fn=make_position_formula_criterion(mlp_predict_fn(position_mlp), MAZE_GEOMETRY))
position_formula_mlp_results = run_benchmark("position_formula_mlp", position_formula_mlp_model, episode_seeds)
position_formula_mlp_results.to_pickle("data/cem_benchmark_position_formula_mlp.pkl")
position_formula_mlp_results["success"].mean()

## Comparison

Aggregates success rate and mean steps-to-goal (successful episodes only) across all seven conditions
(baseline, true_distance x{xgb,mlp}, position_euclidean x{xgb,mlp}, position_formula x{xgb,mlp}).

In [ ]:
all_results = pd.concat(
    [
        baseline_results,
        true_distance_xgb_results, true_distance_mlp_results,
        position_euclidean_xgb_results, position_euclidean_mlp_results,
        position_formula_xgb_results, position_formula_mlp_results,
    ],
    ignore_index=True,
)

summary = all_results.groupby("condition").agg(
    success_rate=("success", "mean"),
    mean_steps_if_success=("steps", lambda s: s[all_results.loc[s.index, "success"]].mean()),
    mean_final_dist=("final_dist", "mean"),
    n_episodes=("success", "count"),
)
summary